# Bronze Quality Profiling — Etapa 2 (itens 2 e 3)

Gera as evidências dos itens **2. Dicionário de dados** e **3. Qualidade dos dados**.

Roda sobre as tabelas **bronze**, e não sobre o silver, porque o `bronze_to_silver`
converte os cinco códigos de ausência (`Recusa`, `Não informou`, `Não sabe informar`,
`Não sabe`, `-`) em um único NULL. Depois dessa conversão não é mais possível medir
*que tipo* de ausência cada município produziu — que é justamente a evidência pedida.

**Saídas**
- `dq_column_profile` — uma linha por coluna: tipo, domínio, completude (esqueleto do item 2)
- `dq_dimension_summary` — uma linha por dimensão de qualidade (tabela 3.7 do documento)
- `dq_nonresponse_by_size` — não-resposta por faixa populacional (evidência de viés)
- `dq_skiplogic_candidates` — pares pai/filha candidatos a filtro de salto (acurácia)

Nada é gravado enquanto o widget `persistir_tabelas` estiver em `nao`.

In [0]:
# Notebook parameters. They appear as input boxes at the top of the notebook.
# Defaults are safe: nothing is written to Unity Catalog until persistir_tabelas = "sim".
dbutils.widgets.dropdown("persistir_tabelas", "nao", ["sim", "nao"], "Gravar tabelas no catálogo?")
dbutils.widgets.text("limiar_nao_resposta", "50", "Descartar acima de X% de não-resposta")
dbutils.widgets.text("limiar_concentracao", "97", "Descartar acima de X% em um único valor")

# Read the widget defensively: the value is whatever the widget held when THIS cell
# ran, so changing the dropdown in the UI does nothing until the cell is re-run.
# Printing the raw string makes that obvious instead of silently behaving as "nao".
_persistir_raw = dbutils.widgets.get("persistir_tabelas")
PERSISTIR = _persistir_raw.strip().lower() in ("sim", "s", "true", "yes", "1")
LIMIAR_NAO_RESPOSTA = float(dbutils.widgets.get("limiar_nao_resposta"))
LIMIAR_CONCENTRACAO = float(dbutils.widgets.get("limiar_concentracao"))

print(f"Widget raw value   : {_persistir_raw!r}")
print(f"Persist tables     : {PERSISTIR}")
print(f"Non-response cutoff: {LIMIAR_NAO_RESPOSTA}%")
print(f"Concentration cutoff: {LIMIAR_CONCENTRACAO}%")

In [0]:
%run ../environment/common_env_variables

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

import pandas as pd
import numpy as np

In [0]:
# Source tables (bronze layer, written by landing_to_bronze)
bronze_tbl_dict = f'{catalog_name}.{bronze_schema}.munic_raw_dict'
bronze_tbl_info = f'{catalog_name}.{bronze_schema}.munic_raw_info_data'
bronze_tbl_gov = f'{catalog_name}.{bronze_schema}.munic_raw_gov_data'

# Output tables. They live in gold with a dq_ prefix so the quality artifacts are
# separated from the analytical product by naming, without creating a new schema
# (which would need CREATE SCHEMA permission).
dq_tbl_profile = f'{catalog_name}.{gold_schema}.dq_column_profile'
dq_tbl_summary = f'{catalog_name}.{gold_schema}.dq_dimension_summary'
dq_tbl_bysize = f'{catalog_name}.{gold_schema}.dq_nonresponse_by_size'
dq_tbl_skip = f'{catalog_name}.{gold_schema}.dq_skiplogic_candidates'

# The five absence codes used by IBGE, kept SEPARATE on purpose.
# "-" is structurally different from the other four: it means the question did not
# apply to this municipality (questionnaire skip logic), not that the municipality
# failed to answer. Collapsing them is a modelling decision for silver; measuring
# them apart is the whole point of this notebook.
COD_NAO_SABE = ["Não sabe informar", "Não sabe"]
COD_NAO_INFORMOU = ["Não informou"]
COD_RECUSA = ["Recusa"]
COD_NAO_APLICAVEL = ["-"]

# Identification columns shared by both worksheets; they are not survey indicators
# and must be excluded from the per-indicator profiling.
COLS_IDENTIFICACAO = [
    "cod_munic", "uf", "cod_uf", "desc_mun",
    "populacao", "faixa_populacao", "regiao",
]
# Columns added by the bronze ingestion job (lineage, not survey content).
COLS_METADADOS = [
    "source_file", "source_url", "dataset_name",
    "reference_year", "ingestion_timestamp",
]

## Parte 1 — Inventário das tabelas bronze

Contagens básicas que alimentam o item 1 do documento e servem de linha de base
para as verificações seguintes.

In [0]:
df_raw_info = spark.table(bronze_tbl_info)
df_raw_gov = spark.table(bronze_tbl_gov)
df_raw_dict = spark.table(bronze_tbl_dict)

# Indicator columns = everything that is neither identification nor ingestion metadata
cols_ind_info = [c for c in df_raw_info.columns
                 if c not in COLS_IDENTIFICACAO + COLS_METADADOS]
cols_ind_gov = [c for c in df_raw_gov.columns
                if c not in COLS_IDENTIFICACAO + COLS_METADADOS]

# Fail loudly if the bronze schema changed in a way that breaks the prefix assumption
assert cols_ind_info, "No indicator columns found in the info table"
assert cols_ind_gov, "No indicator columns found in the gov table"

n_info = df_raw_info.count()
n_gov = df_raw_gov.count()

print(f"munic_raw_info_data: {n_info:,} rows | {len(df_raw_info.columns)} columns "
      f"({len(cols_ind_info)} indicators)")
print(f"munic_raw_gov_data : {n_gov:,} rows | {len(df_raw_gov.columns)} columns "
      f"({len(cols_ind_gov)} indicators)")
print(f"munic_raw_dict     : {df_raw_dict.count():,} rows")

## Parte 2 — Formato longo e classificação da ausência

Em vez de percorrer 374 colunas uma a uma, as duas tabelas são convertidas para
formato longo — uma linha por (município, coluna, valor). Com isso, todas as
métricas das seis dimensões saem de agregações sobre um único DataFrame.

A função `classificar_ausencia` é a **única** definição de "o que conta como ausente"
no projeto: o mesmo critério alimenta o relatório de qualidade e a justificativa do
tratamento aplicado no silver.

In [0]:
def to_long(df, value_cols, tabela_nome):
    """
    Reshape a wide DataFrame into (tabela, cod_munic, coluna, valor).

    Uses Spark's stack() so the whole reshape is a single projection instead of a
    union of one SELECT per column. All values are cast to string because the
    worksheets mix text and numeric columns and we only care about the raw answer.

    Parameters:
        df (DataFrame)   : bronze table
        value_cols (list): indicator column names to unpivot
        tabela_nome (str): label kept in the output so both worksheets can share
                           one long DataFrame

    Returns:
        DataFrame with columns: tabela, cod_munic, coluna, valor
    """
    # stack(n, 'col1', CAST(col1 AS STRING), 'col2', CAST(col2 AS STRING), ...)
    pares = ", ".join([f"'{c}', CAST(`{c}` AS STRING)" for c in value_cols])
    stack_expr = f"stack({len(value_cols)}, {pares}) as (coluna, valor)"

    return (
        df.select(
            F.lit(tabela_nome).alias("tabela"),
            F.col("cod_munic"),
            F.expr(stack_expr),
        )
    )


df_long_info = to_long(df_raw_info, cols_ind_info, "informatica_comunicacao")
df_long_gov = to_long(df_raw_gov, cols_ind_gov, "governanca")
df_long = df_long_info.unionByName(df_long_gov)

print(f"Long format: {len(cols_ind_info) + len(cols_ind_gov)} columns "
      f"x {n_info:,} municipalities")

In [0]:
def classificar_ausencia(col_valor):
    """
    Classify a raw survey answer into one of six mutually exclusive classes.

    preenchido     -> an actual answer
    nao_sabe       -> respondent does not know ("Não sabe informar" / "Não sabe")
    nao_informou   -> respondent did not provide the information
    recusa         -> respondent refused
    nao_aplicavel  -> "-", question skipped by questionnaire routing
    nulo           -> NULL or empty string in the source file

    The last five are all "absent", but they are NOT the same thing, and the report
    has to tell them apart. Returns a Column expression, not a value.
    """
    valor = F.trim(col_valor)
    return (
        F.when(col_valor.isNull(), "nulo")
        .when(valor == "", "nulo")
        .when(valor.isin(COD_NAO_SABE), "nao_sabe")
        .when(valor.isin(COD_NAO_INFORMOU), "nao_informou")
        .when(valor.isin(COD_RECUSA), "recusa")
        .when(valor.isin(COD_NAO_APLICAVEL), "nao_aplicavel")
        .otherwise("preenchido")
    )


df_long = df_long.withColumn("classe", classificar_ausencia(F.col("valor")))

# No .cache() here on purpose: serverless compute rejects PERSIST/CACHE
# (NOT_SUPPORTED_WITH_SERVERLESS). The stack() reshape is cheap enough to recompute,
# and serverless applies its own disk caching underneath.

display(
    df_long.groupBy("classe").count()
    .withColumn("pct", F.round(F.col("count") / F.sum("count").over(Window.partitionBy()) * 100, 2))
    .orderBy(F.desc("count"))
)

## Item 2 — Dicionário de dados

Gera o esqueleto da tabela do item 2: **Campo**, **Tipo**, **Domínio** e **Qualidade**
saem do perfilamento; **Significado** vem da tabela de dicionário do bronze, quando o
código existe lá.

In [0]:
# One aggregation over the long DataFrame produces every per-column metric at once.
# F.collect_set ignores NULLs, so wrapping the value in F.when(...) keeps only the
# answers that are actually filled in.
df_profile = (
    df_long.groupBy("tabela", "coluna")
    .agg(
        F.count("*").alias("n_registros"),
        F.sum(F.when(F.col("classe") == "preenchido", 1).otherwise(0)).alias("n_preenchido"),
        F.sum(F.when(F.col("classe") == "nao_sabe", 1).otherwise(0)).alias("n_nao_sabe"),
        F.sum(F.when(F.col("classe") == "nao_informou", 1).otherwise(0)).alias("n_nao_informou"),
        F.sum(F.when(F.col("classe") == "recusa", 1).otherwise(0)).alias("n_recusa"),
        F.sum(F.when(F.col("classe") == "nao_aplicavel", 1).otherwise(0)).alias("n_nao_aplicavel"),
        F.sum(F.when(F.col("classe") == "nulo", 1).otherwise(0)).alias("n_nulo"),
        F.collect_set(
            F.when(F.col("classe") == "preenchido", F.col("valor"))
        ).alias("valores_distintos"),
    )
)

# Derived columns: percentages, detected type and a readable domain sample.
df_profile = (
    df_profile
    # Non-response = the respondent could have answered and did not.
    # "nao_aplicavel" is deliberately OUT of this sum: the question never applied.
    .withColumn("n_nao_resposta",
                F.col("n_nao_sabe") + F.col("n_nao_informou") + F.col("n_recusa") + F.col("n_nulo"))
    .withColumn("pct_preenchido", F.round(F.col("n_preenchido") / F.col("n_registros") * 100, 2))
    .withColumn("pct_nao_resposta", F.round(F.col("n_nao_resposta") / F.col("n_registros") * 100, 2))
    .withColumn("pct_nao_aplicavel", F.round(F.col("n_nao_aplicavel") / F.col("n_registros") * 100, 2))
    .withColumn("n_distintos", F.size("valores_distintos"))
    # Type detection, in the same spirit as treat_survey_indicators in bronze_to_silver:
    # forall() tests whether EVERY distinct value is Sim or Não.
    .withColumn(
        "tipo_detectado",
        F.when(F.col("n_distintos") == 0, "sem_resposta")
        .when(F.expr("forall(valores_distintos, x -> x in ('Sim', 'Não'))"), "binario")
        .when(F.col("n_distintos") <= 20, "categorico")
        .otherwise("texto_livre"),
    )
    # Domain sample: up to 6 distinct values, alphabetically, pipe-separated.
    # Free-text columns would blow up the cell, so they are labelled instead.
    .withColumn(
        "dominio_amostra",
        F.when(F.col("tipo_detectado") == "texto_livre",
               F.concat(F.lit("texto livre ("), F.col("n_distintos").cast("string"), F.lit(" valores distintos)")))
        .otherwise(F.concat_ws(" | ", F.slice(F.array_sort("valores_distintos"), 1, 6))),
    )
)

In [0]:
# A column where 99% of municipalities give the same answer carries almost no
# information for clustering, even when it is 100% complete. This is the second
# filter of the selection funnel.
w_col = Window.partitionBy("tabela", "coluna").orderBy(F.desc("n"), F.asc("valor"))

df_concentracao = (
    df_long.filter(F.col("classe") == "preenchido")
    .groupBy("tabela", "coluna", "valor").agg(F.count("*").alias("n"))
    .withColumn("rk", F.row_number().over(w_col))
    .filter(F.col("rk") == 1)
    .select(
        "tabela", "coluna",
        F.col("valor").alias("valor_mais_frequente"),
        F.col("n").alias("n_valor_mais_frequente"),
    )
)

df_profile = (
    df_profile.join(df_concentracao, on=["tabela", "coluna"], how="left")
    .withColumn(
        "pct_concentracao",
        F.round(F.col("n_valor_mais_frequente") / F.col("n_preenchido") * 100, 2),
    )
)

In [0]:
# The dictionary stores codes uppercase (MGOV32) while bronze lowercased the column
# names (mgov32). Joining on lower(cod_variable) is what reconciles the two — and the
# fact that this is needed at all is itself a finding for the documentation section.
df_dict_lookup = (
    df_raw_dict
    .select(F.lower(F.col("cod_variable")).alias("coluna"), F.col("meaning").alias("significado"))
    .dropDuplicates(["coluna"])
)

df_profile = df_profile.join(df_dict_lookup, on="coluna", how="left")

# Seven columns exist in the data but are absent from the IBGE dictionary sheet.
# Their meaning was reconstructed from the observed values and from the parent
# question, and each reconstruction is recorded here so the documentation can state
# exactly which definitions are the group's and which came from IBGE.
SIGNIFICADOS_MANUAIS = {
    "mtic11": "Endereço eletrônico (URL) da página da prefeitura na internet",
    "mtic31": "Denominação da estrutura organizacional da área de comunicação",
    "mtic331": "Denominação do cargo ou carreira específica de comunicação social",
    "mtic2421": "Abrangência da conexão Wi-Fi na área urbana (parte ou toda)",
    "mtic2431": "Abrangência da conexão Wi-Fi na área rural (parte ou toda)",
    "mgov141": "Total de pessoal ocupado especificamente no controle interno (quantidade informada)",
    "mgov142": "Não sabe informar o total de pessoal ocupado no controle interno",
}

# How many columns the IBGE dictionary failed to document (evidence for item 2.1),
# measured BEFORE the manual meanings are applied.
n_sem_significado = df_profile.filter(F.col("significado").isNull()).count()

# Build a Spark map literal and coalesce it over the dictionary meaning.
mapa_manual = F.create_map(
    *[x for k, v in SIGNIFICADOS_MANUAIS.items() for x in (F.lit(k), F.lit(v))]
)

df_profile = (
    df_profile
    .withColumn(
        "origem_significado",
        F.when(F.col("significado").isNotNull(), "dicionário IBGE")
        .when(F.col("coluna").isin(list(SIGNIFICADOS_MANUAIS)), "inferido pelo grupo")
        .otherwise("não documentado"),
    )
    .withColumn("significado", F.coalesce(F.col("significado"), mapa_manual[F.col("coluna")]))
)

n_colunas = df_profile.count()
n_ainda_sem_significado = df_profile.filter(F.col("significado").isNull()).count()
n_dict_total = df_dict_lookup.count()
n_dict_orfao = (
    df_dict_lookup.join(df_profile.select("coluna"), on="coluna", how="left_anti").count()
)

print(f"Columns profiled                      : {n_colunas}")
print(f"Columns with no IBGE dictionary entry : {n_sem_significado}")
print(f"  of which filled in manually         : {len(SIGNIFICADOS_MANUAIS)}")
print(f"  still undocumented                  : {n_ainda_sem_significado}")
print(f"Dictionary codes with no column       : {n_dict_orfao} (out of {n_dict_total})")

display(
    df_profile.filter(F.col("origem_significado") == "inferido pelo grupo")
    .select("tabela", "coluna", "significado", "dominio_amostra", "pct_nao_aplicavel")
)

In [0]:
# This is the table that goes into the Word document. The only column that still
# needs manual work is "significado" where the dictionary had no entry.
df_dicionario = (
    df_profile
    .select(
        F.col("coluna").alias("campo"),
        F.col("significado"),
        F.col("tipo_detectado").alias("tipo"),
        F.col("dominio_amostra").alias("dominio"),
        F.concat(
            F.col("pct_nao_resposta").cast("string"), F.lit("% não-resposta; "),
            F.col("pct_nao_aplicavel").cast("string"), F.lit("% não aplicável"),
        ).alias("qualidade"),
        "tabela",
        F.col("origem_significado").alias("origem_do_significado"),
    )
    .orderBy("tabela", "campo")
)

display(df_dicionario)

## Item 3 — Qualidade dos dados

Uma seção por dimensão. Cada uma produz um número que vai direto para o documento.

In [0]:
# Overall completeness, then the worst and best columns.
tot = df_profile.agg(
    F.sum("n_preenchido").alias("preenchido"),
    F.sum("n_nao_resposta").alias("nao_resposta"),
    F.sum("n_nao_aplicavel").alias("nao_aplicavel"),
    F.sum("n_registros").alias("total"),
).collect()[0]

pct_global_preenchido = round(tot["preenchido"] / tot["total"] * 100, 2)
pct_global_nao_resposta = round(tot["nao_resposta"] / tot["total"] * 100, 2)
pct_global_nao_aplicavel = round(tot["nao_aplicavel"] / tot["total"] * 100, 2)

print(f"Filled        : {pct_global_preenchido}%")
print(f"Non-response  : {pct_global_nao_resposta}%")
print(f"Not applicable: {pct_global_nao_aplicavel}%")

print("\nWorst 10 columns by non-response:")
display(
    df_profile.select("tabela", "coluna", "significado", "pct_nao_resposta", "pct_nao_aplicavel")
    .orderBy(F.desc("pct_nao_resposta")).limit(10)
)

In [0]:
# Validity here = does the observed domain match what the column claims to be.
# Binary columns must hold only Sim/Não; anything else is a domain violation.
# Columns whose only content is absence are flagged as structurally empty.
df_validade = (
    df_profile
    .withColumn(
        "achado_validade",
        F.when(F.col("tipo_detectado") == "sem_resposta", "coluna sem nenhuma resposta válida")
        .when((F.col("tipo_detectado") == "categorico") & (F.col("n_distintos") == 1),
              "um único valor observado")
        .when(F.col("pct_nao_aplicavel") > 50, "maioria não aplicável (filtro de salto)")
        .otherwise(None),
    )
    .filter(F.col("achado_validade").isNotNull())
    .select("tabela", "coluna", "tipo_detectado", "n_distintos",
            "pct_nao_aplicavel", "achado_validade")
)

n_validade = df_validade.count()
print(f"Columns with a validity finding: {n_validade}")
display(df_validade.orderBy(F.desc("pct_nao_aplicavel")))

In [0]:
# .distinct() is mandatory: .select() alone does not deduplicate, and without it the
# check would just recount the rows and always "pass".
n_info_distinct = df_raw_info.select("cod_munic").distinct().count()
n_gov_distinct = df_raw_gov.select("cod_munic").distinct().count()

print(f"info: {n_info:,} rows / {n_info_distinct:,} distinct cod_munic")
print(f"gov : {n_gov:,} rows / {n_gov_distinct:,} distinct cod_munic")

unicidade_ok = (n_info == n_info_distinct) and (n_gov == n_gov_distinct)
print(f"\nPrimary key unique in both tables: {unicidade_ok}")

In [0]:
# bronze_to_silver takes the municipality descriptors from the info worksheet only,
# assuming the gov worksheet repeats them identically. Nobody had verified that.
cols_comparar = ["uf", "populacao", "faixa_populacao", "regiao"]

df_desc_info = df_raw_info.select("cod_munic", *[F.col(c).alias(f"{c}_info") for c in cols_comparar])
df_desc_gov = df_raw_gov.select("cod_munic", *[F.col(c).alias(f"{c}_gov") for c in cols_comparar])

df_desc = df_desc_info.join(df_desc_gov, on="cod_munic", how="inner")

divergencias = {}
for c in cols_comparar:
    divergencias[c] = df_desc.filter(
        ~F.col(f"{c}_info").eqNullSafe(F.col(f"{c}_gov"))
    ).count()

for c, n in divergencias.items():
    print(f"{c:18s}: {n} municipalities differ between the two worksheets")

n_divergencias = sum(divergencias.values())

# Key coverage: do both worksheets describe exactly the same municipalities?
n_so_info = df_raw_info.select("cod_munic").subtract(df_raw_gov.select("cod_munic")).count()
n_so_gov = df_raw_gov.select("cod_munic").subtract(df_raw_info.select("cod_munic")).count()
print(f"\nOnly in info: {n_so_info} | Only in gov: {n_so_gov}")

In [0]:
# The lineage columns written by landing_to_bronze answer this dimension directly.
df_lineage = df_raw_gov.select(
    *[c for c in COLS_METADADOS if c in df_raw_gov.columns]
).distinct()

display(df_lineage)

lineage = df_lineage.collect()[0].asDict() if df_lineage.count() == 1 else {}
ref_year = lineage.get("reference_year", "?")
ingestion = lineage.get("ingestion_timestamp", None)

dias_desde_ingestao = (
    spark.sql(f"SELECT datediff(current_date(), date('{ingestion}')) AS d").collect()[0]["d"]
    if ingestion else None
)
print(f"\nReference year: {ref_year} | days since ingestion: {dias_desde_ingestao}")

### 3.6 Acurácia — detecção empírica de filtros de salto

Não há fonte externa para confrontar as respostas, então a aproximação possível é
a **coerência interna**: se a pergunta B só se aplica quando a pergunta A foi "Sim",
então "A = Não" deveria implicar "B = `-`".

O bloco abaixo procura, para cada coluna com muito `-`, qual coluna binária melhor
explica esse padrão. O resultado é **hipótese**, não conclusão: serve para você
confirmar contra o questionário da MUNIC e então reportar as incoerências.

Três cuidados de desenho, cada um corrigindo um modo de falhar observado na base:

1. **O pai nem sempre é binário.** `Mgov09` tem três respostas e o bloco abaixo dele é
   pulado em duas. Restringir a busca a pais Sim/Não faz o detector descartar o pai
   verdadeiro e eleger uma coluna apenas correlacionada.
2. **O gatilho pode ser um conjunto de valores.** `Mgov33` é pulada quando `Mgov32` é
   "Não" **ou** "Não sabe informar". Testar um valor por vez só explica parte dos casos.
3. **O gatilho tem de ser uma resposta.** Se `-` pudesse disparar o salto, o detector
   elegeria um irmão do mesmo bloco — duas colunas que ficam vazias juntas por
   consequência compartilhada, não por causa. Códigos de não-resposta, ao contrário,
   são gatilhos legítimos.

Pares com cobertura ou precisão abaixo de 95% são marcados como `parcial`: não são
incoerências, e sim variáveis cuja não aplicabilidade decorre de uma cadeia de mais de
um nível, que este procedimento não percorre.

In [0]:
# Two groups of columns are involved:
#   parents  -> columns with few distinct values (binary OR short categorical)
#   children -> columns with a meaningful share of "-"
#
# The parent does NOT have to be binary. Mgov09 ("estrutura de controle interno")
# has three answers, and the block below it is skipped on two of them, so a
# binary-only search would miss it and settle for a correlated but unrelated column.
MIN_PCT_NAO_APLICAVEL = 5.0
MAX_VALORES_PAI = 8
MIN_CONFIRMACAO = 95.0   # cobertura and precisao, in %

# The trigger must be an actual ANSWER. If "-" were allowed to trigger, the detector
# would elect a sibling from the same block — two columns that go blank together,
# which is a shared consequence, not a cause. Non-response codes ARE allowed:
# "Não sabe informar" legitimately skips the follow-up question.
NAO_PODE_SER_GATILHO = {"-", "None", "nan", ""}

pdf_prof = df_profile.select(
    "tabela", "coluna", "tipo_detectado", "pct_nao_aplicavel", "n_distintos", "significado",
).toPandas()

cols_filha = pdf_prof.loc[pdf_prof["pct_nao_aplicavel"] >= MIN_PCT_NAO_APLICAVEL, "coluna"].tolist()
cols_pai = pdf_prof.loc[pdf_prof["n_distintos"] <= MAX_VALORES_PAI, "coluna"].tolist()

print(f"Candidate parents (<= {MAX_VALORES_PAI} distinct values): {len(cols_pai)}")
print(f"Candidate children (>= {MIN_PCT_NAO_APLICAVEL}% not applicable): {len(cols_filha)}")

df_skip_candidates = None

if cols_pai and cols_filha:
    pdf_long = (
        df_long.filter(F.col("coluna").isin(list(set(cols_pai + cols_filha))))
        .select("cod_munic", "coluna", "valor")
        .toPandas()
    )
    mat = pdf_long.pivot(index="cod_munic", columns="coluna", values="valor").astype(str)

    # Y[i, j] = municipality i was not applicable on child j
    Y = mat.reindex(columns=cols_filha).eq("-").to_numpy()
    n_y = Y.sum(axis=0)

    melhor_f1 = np.zeros(len(cols_filha))
    melhor = [None] * len(cols_filha)

    for pai in cols_pai:
        valores = sorted(mat[pai].unique())
        validos = np.array([v not in NAO_PODE_SER_GATILHO for v in valores])
        if not validos.any():
            continue

        # One-hot of the parent's answers: M[k, i] = municipality i answered valores[k]
        M = np.stack([(mat[pai] == v).to_numpy() for v in valores])
        tot = M.sum(axis=1)                                  # how often each answer occurs
        inter = M.astype(np.int32) @ Y.astype(np.int32)      # answer x child co-occurrence

        with np.errstate(divide="ignore", invalid="ignore"):
            taxa = np.where(tot[:, None] > 0, inter / tot[:, None], 0.0)

        # The trigger is the SET of answers under which the child is mostly "-".
        # Picking a set instead of a single value is what captures composite skips
        # such as "Não" OR "Não sabe informar".
        G = (taxa > 0.5) & validos[:, None]

        n_gatilho = (G * tot[:, None]).sum(axis=0)
        n_acerto = (G * inter).sum(axis=0)

        with np.errstate(divide="ignore", invalid="ignore"):
            cobertura = np.where(n_y > 0, n_acerto / n_y, 0.0)
            precisao = np.where(n_gatilho > 0, n_acerto / n_gatilho, 0.0)

        f1 = np.where((cobertura + precisao) > 0,
                      2 * cobertura * precisao / (cobertura + precisao), 0.0)

        for j in np.nonzero(f1 > melhor_f1)[0]:
            if cols_filha[j] == pai:
                continue
            melhor_f1[j] = f1[j]
            melhor[j] = {
                "coluna_filha": cols_filha[j],
                "coluna_pai": pai,
                "valores_gatilho": " | ".join(valores[k] for k in range(len(valores)) if G[k, j]),
                "pct_cobertura": round(float(cobertura[j]) * 100, 2),
                "pct_precisao": round(float(precisao[j]) * 100, 2),
                "n_nao_aplicavel": int(n_y[j]),
                "hipotese_confirmada": None,   # fill in after checking the questionnaire
            }

    pdf_skip = pd.DataFrame([m for m in melhor if m])
    pdf_skip["relacao_confirmada"] = np.where(
        (pdf_skip.pct_cobertura >= MIN_CONFIRMACAO) & (pdf_skip.pct_precisao >= MIN_CONFIRMACAO),
        "sim", "parcial",
    )
    pdf_skip = pdf_skip.sort_values(
        ["pct_cobertura", "pct_precisao"], ascending=False
    ).reset_index(drop=True)

    n_conf = int((pdf_skip.relacao_confirmada == "sim").sum())
    n_exato = int(((pdf_skip.pct_cobertura == 100) & (pdf_skip.pct_precisao == 100)).sum())
    n_composto = int(pdf_skip.valores_gatilho.str.contains(r"\|").sum())

    print(f"\nChildren evaluated     : {len(pdf_skip)}")
    print(f"Relationship confirmed : {n_conf} ({n_conf / len(pdf_skip) * 100:.0f}%)")
    print(f"  exact (100/100)      : {n_exato}")
    print(f"Composite triggers     : {n_composto}")
    print(f"Partial (multi-level)  : {len(pdf_skip) - n_conf}")

    df_skip_candidates = spark.createDataFrame(pdf_skip)
    display(df_skip_candidates)
else:
    print("Not enough candidates to run the detector.")

## Viés e grupos sub-representados

Se a ausência de resposta estiver correlacionada com o porte do município, ela não
é aleatória — e tratá-la como se fosse enviesa toda a solução.

**Atenção à unidade de análise.** Medir a não-resposta sobre as 360 colunas de uma vez
dilui o sinal: em quase todas elas a não-resposta é próxima de zero, e a média acaba
dizendo que o porte quase não importa. O viés que interessa está no bloco de LGPD,
onde está a variável-alvo do projeto. As duas medidas estão abaixo, nessa ordem:
a agregada como **contexto**, a do bloco LGPD como **evidência**.

In [0]:
df_faixa = df_raw_info.select("cod_munic", "faixa_populacao")

df_por_faixa = (
    df_long.join(df_faixa, on="cod_munic", how="inner")
    .groupBy("faixa_populacao")
    .agg(
        F.count("*").alias("n_celulas"),
        F.sum(F.when(F.col("classe").isin("nao_sabe", "nao_informou", "recusa", "nulo"), 1)
              .otherwise(0)).alias("n_nao_resposta"),
        F.countDistinct("cod_munic").alias("n_municipios"),
    )
    .withColumn("pct_nao_resposta",
                F.round(F.col("n_nao_resposta") / F.col("n_celulas") * 100, 2))
    .orderBy("faixa_populacao")
)

display(df_por_faixa)

In [0]:
# The LGPD questions are the ones this project is built on, and they are the four
# worst columns in the whole dataset. Measured on their own, the relationship with
# municipality size is the opposite of what the aggregate above suggests.
COLS_LGPD = ["mgov32", "mgov33", "mgov34", "mgov35", "mgov36"]
COL_ALVO = "mgov32"

df_lgpd_faixa = (
    df_long.filter(F.col("coluna").isin(COLS_LGPD))
    .join(df_faixa, on="cod_munic", how="inner")
    .groupBy("faixa_populacao")
    .agg(
        F.count("*").alias("n_celulas"),
        F.sum(F.when(F.col("classe").isin("nao_sabe", "nao_informou", "recusa", "nulo"), 1)
              .otherwise(0)).alias("n_nao_resposta"),
    )
    .withColumn("pct_nao_resposta",
                F.round(F.col("n_nao_resposta") / F.col("n_celulas") * 100, 2))
)

# Same cut for the target variable alone, plus the share that answered "Sim".
# The two together are the whole argument: small municipalities both answer less
# AND answer "Sim" less, so the missing answers are not missing at random.
df_alvo_faixa = (
    df_long.filter(F.col("coluna") == COL_ALVO)
    .join(df_faixa, on="cod_munic", how="inner")
    .groupBy("faixa_populacao")
    .agg(
        F.count("*").alias("n_municipios"),
        F.sum(F.when(F.col("classe") != "preenchido", 1).otherwise(0)).alias("n_sem_resposta_alvo"),
        F.sum(F.when(F.col("valor") == "Sim", 1).otherwise(0)).alias("n_sim"),
    )
    .withColumn("pct_sem_resposta_alvo",
                F.round(F.col("n_sem_resposta_alvo") / F.col("n_municipios") * 100, 2))
    .withColumn("pct_sim",
                F.round(F.col("n_sim") / F.col("n_municipios") * 100, 2))
)

df_vies_faixa = (
    df_lgpd_faixa.select("faixa_populacao", F.col("pct_nao_resposta").alias("pct_nr_bloco_lgpd"))
    .join(
        df_alvo_faixa.select("faixa_populacao", "n_municipios",
                             "pct_sem_resposta_alvo", "pct_sim"),
        on="faixa_populacao", how="inner",
    )
    .orderBy("faixa_populacao")
)

display(df_vies_faixa)

In [0]:
# Sub-representation is about how many municipalities sit in each band.
df_representacao = (
    df_raw_info.groupBy("faixa_populacao").agg(F.count("*").alias("n_municipios"))
    .withColumn("pct_do_total", F.round(F.col("n_municipios") / n_info * 100, 2))
    .orderBy("faixa_populacao")
)

display(df_representacao)

## Funil de seleção de variáveis

O recorte do projeto deixa de ser escolha subjetiva e passa a ser resultado de
critérios explícitos, aplicados nesta ordem e com a contagem de descarte de cada um.

In [0]:
df_funil = (
    df_profile
    .withColumn(
        "status_funil",
        F.when(F.col("tipo_detectado") == "sem_resposta", "descartada: sem resposta válida")
        .when(F.col("pct_nao_resposta") > LIMIAR_NAO_RESPOSTA,
              F.concat(F.lit("descartada: não-resposta > "), F.lit(LIMIAR_NAO_RESPOSTA), F.lit("%")))
        .when(F.col("pct_concentracao") > LIMIAR_CONCENTRACAO,
              F.concat(F.lit("descartada: concentração > "), F.lit(LIMIAR_CONCENTRACAO), F.lit("%")))
        .when(F.col("tipo_detectado") == "texto_livre", "descartada: texto livre")
        .otherwise("mantida"),
    )
)

display(df_funil.groupBy("status_funil").count().orderBy(F.desc("count")))

n_mantidas = df_funil.filter(F.col("status_funil") == "mantida").count()
print(f"\n{n_mantidas} of {n_colunas} columns survive the funnel")

# The profile table carries the funnel status so the decision travels with the data
df_profile = df_funil

In [0]:
# One row per quality dimension: what was checked, the number, and the verdict.
# This is the table you paste into the Word document.
linhas_resumo = [
    ("Completude",
     "Percentual de células preenchidas, de não-resposta e de não aplicável sobre as 374 colunas",
     f"{pct_global_preenchido}% preenchido; {pct_global_nao_resposta}% não-resposta; "
     f"{pct_global_nao_aplicavel}% não aplicável",
     "Ausência relevante e concentrada em colunas específicas"),

    ("Validade",
     "Aderência dos valores ao domínio esperado e identificação de colunas sem resposta válida",
     f"{n_validade} colunas com achado de validade",
     "Requer tratamento documentado antes do uso"),

    ("Consistência",
     "Comparação dos descritores municipais entre as duas abas e cobertura do dicionário",
     f"{n_divergencias} divergências de descritores entre as abas; {n_sem_significado} colunas "
     f"sem entrada no dicionário do IBGE ({len(SIGNIFICADOS_MANUAIS)} reconstruídas pelo grupo, "
     f"{n_ainda_sem_significado} ainda sem definição); {n_dict_orfao} códigos do dicionário sem "
     f"coluna correspondente",
     "Divergência de nomenclatura entre documentação e dados"),

    ("Unicidade",
     "Contagem de valores distintos da chave cod_munic em cada aba",
     f"info: {n_info_distinct:,} de {n_info:,}; gov: {n_gov_distinct:,} de {n_gov:,}",
     "Chave íntegra" if unicidade_ok else "Duplicidade detectada"),

    ("Atualidade",
     "Defasagem entre ano de referência, data de ingestão e data de uso",
     f"ano de referência {ref_year}; {dias_desde_ingestao} dias desde a ingestão",
     "Dado anual; defasagem compatível com o uso pretendido"),

    ("Acurácia",
     "Coerência interna entre colunas encadeadas por filtro de salto do questionário",
     f"{0 if df_skip_candidates is None else df_skip_candidates.count()} pares pai/filha "
     f"candidatos identificados; confirmação pendente no questionário",
     "Aproximação por coerência interna; sem fonte externa de confronto"),
]

df_resumo = spark.createDataFrame(
    linhas_resumo, schema=["dimensao", "verificacao", "evidencia", "avaliacao"]
)

display(df_resumo)

## Gráficos (figuras do documento)

In [0]:
import matplotlib.pyplot as plt

pdf_vies = df_vies_faixa.toPandas()

# Single series: no legend needed, the title names it. One accent colour, recessive
# axes, direct labels on the bars instead of a value on every gridline.
ACCENT = "#2F6F9F"
INK = "#3C3C3C"

fig, ax = plt.subplots(figsize=(9, 4.5), dpi=150)
barras = ax.bar(pdf_vies["faixa_populacao"], pdf_vies["pct_nr_bloco_lgpd"],
                color=ACCENT, width=0.62)

for b, v in zip(barras, pdf_vies["pct_nr_bloco_lgpd"]):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.25, f"{v:.1f}%",
            ha="center", va="bottom", fontsize=9, color=INK)

ax.set_title("Não-resposta no bloco de LGPD, por faixa populacional",
             fontsize=12, color=INK, pad=12)
ax.set_ylabel("% de células sem resposta", fontsize=10, color=INK)
ax.tick_params(axis="x", rotation=30, labelsize=8, colors=INK)
ax.tick_params(axis="y", labelsize=9, colors=INK)
ax.grid(axis="y", color="#E4E4E4", linewidth=0.8)
ax.set_axisbelow(True)
for lado in ["top", "right", "left"]:
    ax.spines[lado].set_visible(False)
ax.spines["bottom"].set_color("#CCCCCC")
plt.setp(ax.get_xticklabels(), ha="right")
plt.tight_layout()
plt.show()

In [0]:
# Second half of the bias argument: the answer itself follows municipality size.
pdf_alvo = pdf_vies

fig, ax = plt.subplots(figsize=(9, 4.5), dpi=150)
barras = ax.bar(pdf_alvo["faixa_populacao"], pdf_alvo["pct_sim"],
                color=ACCENT, width=0.62)

for b, v in zip(barras, pdf_alvo["pct_sim"]):
    ax.text(b.get_x() + b.get_width() / 2, v + 1.2, f"{v:.1f}%",
            ha="center", va="bottom", fontsize=9, color=INK)

ax.set_title("Municípios com área ou pessoa responsável pela LGPD (mgov32 = Sim)",
             fontsize=12, color=INK, pad=12)
ax.set_ylabel("% dos municípios da faixa", fontsize=10, color=INK)
ax.tick_params(axis="x", rotation=30, labelsize=8, colors=INK)
ax.tick_params(axis="y", labelsize=9, colors=INK)
ax.grid(axis="y", color="#E4E4E4", linewidth=0.8)
ax.set_axisbelow(True)
for lado in ["top", "right", "left"]:
    ax.spines[lado].set_visible(False)
ax.spines["bottom"].set_color("#CCCCCC")
plt.setp(ax.get_xticklabels(), ha="right")
plt.tight_layout()
plt.show()

In [0]:
pdf_rep = df_representacao.toPandas()

fig, ax = plt.subplots(figsize=(9, 4.5), dpi=150)
barras = ax.bar(pdf_rep["faixa_populacao"], pdf_rep["n_municipios"],
                color=ACCENT, width=0.62)

for b, v in zip(barras, pdf_rep["n_municipios"]):
    ax.text(b.get_x() + b.get_width() / 2, v + 12, f"{v:,}".replace(",", "."),
            ha="center", va="bottom", fontsize=9, color=INK)

ax.set_title("Municípios por faixa populacional", fontsize=12, color=INK, pad=12)
ax.set_ylabel("Número de municípios", fontsize=10, color=INK)
ax.tick_params(axis="x", rotation=30, labelsize=8, colors=INK)
ax.tick_params(axis="y", labelsize=9, colors=INK)
ax.grid(axis="y", color="#E4E4E4", linewidth=0.8)
ax.set_axisbelow(True)
for lado in ["top", "right", "left"]:
    ax.spines[lado].set_visible(False)
ax.spines["bottom"].set_color("#CCCCCC")
plt.setp(ax.get_xticklabels(), ha="right")
plt.tight_layout()
plt.show()

## Persistência

Só grava se o widget `persistir_tabelas` estiver em `sim`. Enquanto estiver em `nao`,
o notebook é somente leitura e pode rodar sem permissão de escrita no catálogo.

In [0]:
def gravar(df, nome_tabela, descricao):
    """Write a quality artifact as a Delta table, recording why it exists."""
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .option("userMetadata", descricao)
        .saveAsTable(nome_tabela)
    )
    print(f"Written: {nome_tabela}")


if PERSISTIR:
    # valores_distintos is an array used only to derive the other columns; it would
    # bloat the table without adding anything the profile does not already state.
    gravar(df_profile.drop("valores_distintos"), dq_tbl_profile,
           "Perfil de qualidade por coluna das tabelas bronze da MUNIC 2024")
    gravar(df_resumo, dq_tbl_summary,
           "Sintese das seis dimensoes de qualidade - Etapa 2")
    gravar(df_vies_faixa, dq_tbl_bysize,
           "Nao-resposta no bloco LGPD e taxa de Sim do alvo, por faixa populacional")
    if df_skip_candidates is not None:
        gravar(df_skip_candidates, dq_tbl_skip,
               "Pares pai/filha candidatos a filtro de salto - hipoteses a confirmar")
else:
    print("persistir_tabelas = nao — nothing was written.")
    print("Use the download button on each display() output to export CSV.")

In [0]:
# Writing to the volume needs WRITE VOLUME. If it fails, the display() outputs above
# still have a download button, which is enough to move the numbers into the document.
if PERSISTIR:
    pasta_saida = f"{landing_folder_path}/quality_outputs"
    try:
        dbutils.fs.mkdirs(pasta_saida)
        df_dicionario.toPandas().to_csv(f"{pasta_saida}/dicionario_esqueleto.csv",
                                        index=False, encoding="utf-8-sig")
        df_profile.drop("valores_distintos").toPandas().to_csv(
            f"{pasta_saida}/qualidade_colunas.csv", index=False, encoding="utf-8-sig")
        df_resumo.toPandas().to_csv(f"{pasta_saida}/qualidade_sintese.csv",
                                    index=False, encoding="utf-8-sig")
        # Evidence behind section 3.8 (bias charts) and section 3.6 (skip logic).
        # Without these two the report cites numbers that no attached file carries.
        df_vies_faixa.toPandas().to_csv(f"{pasta_saida}/vies_por_faixa.csv",
                                        index=False, encoding="utf-8-sig")
        if df_skip_candidates is not None:
            df_skip_candidates.toPandas().to_csv(f"{pasta_saida}/filtros_de_salto.csv",
                                                 index=False, encoding="utf-8-sig")
        print(f"CSV files written to {pasta_saida}")
    except Exception as e:
        print(f"CSV export failed ({type(e).__name__}): {e}")
        print("Use the download button on the display() outputs instead.")

In [0]:
print("Profiling finished.")